# 第 9 章 決定木

「どの質問をすれば、もっともよくデータが分かれるか」を貪欲に選び続けて木を育てます。

対応する記事: [第 9 章 決定木（Python 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/python/ch09.md)

実装本体: `apps/grokking-ml-python/src/`

## セットアップ

実装本体（`../src/grokking_ml/`）を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

```bash
cd apps/grokking-ml-python
uv sync
uv run jupyter lab notebooks/
```

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from grokking_ml.ch09_decision_trees import *

## 不純度

**ジニ不純度は「ランダムに 2 つ選んだとき、ラベルが食い違う確率」**、**エントロピーは「ラベルを 1 つ伝えるのに必要なビット数」** と読めます。値の範囲は違いますが、順序は一致します。

In [2]:
print(f"{'集合':<20} {'ジニ':>8} {'エントロピー':>14}")
for name, ls in [("純粋 [1,1,1]", [1, 1, 1]),
                 ("偏り [1,1,1,0]", [1, 1, 1, 0]),
                 ("均等 [1,1,0,0]", [1, 1, 0, 0]),
                 ("4 クラス均等", [0, 1, 2, 3])]:
    print(f"{name:<20} {gini_impurity(ls):>8.4f} {entropy(ls):>14.4f}")

集合                         ジニ         エントロピー
純粋 [1,1,1]             0.0000        -0.0000
偏り [1,1,1,0]           0.3750         0.8113
均等 [1,1,0,0]           0.5000         1.0000
4 クラス均等                0.7500         2.0000


## データセット

原著と同じアプリ推薦データです。特徴量は性別（0=女性、1=男性）と年齢。**「若い人には推薦する」という規則が隠れていますが、性別は関係ありません。**

In [3]:
points = [(1.0, 15.0), (0.0, 25.0), (0.0, 32.0), (1.0, 35.0),
          (0.0, 12.0), (1.0, 14.0), (1.0, 55.0), (0.0, 40.0)]
labels = [1, 0, 0, 0, 1, 1, 0, 0]

print(f"根の不純度 ジニ {gini_impurity(labels):.4f} / エントロピー {entropy(labels):.4f}")

根の不純度 ジニ 0.4688 / エントロピー 0.9544


## すべての分割候補を評価する

**決定木は自力で「年齢が効く、性別は関係ない」を見つけます。** 「年齢 < 20」の利得が根の不純度と一致し、この 1 回の質問で不純度が 0 になることが分かります。

In [4]:
print(f"{'特徴量':<8} {'閾値':>8} {'情報利得':>10}")
for split in candidate_splits(points):
    _, left, _, right = apply_split(points, labels, split)
    if not left or not right:
        continue
    name = "性別" if split.feature == 0 else "年齢"
    print(f"{name:<8} {split.threshold:>8.1f} {information_gain(labels, left, right):>10.4f}")

特徴量            閾値       情報利得
性別            0.5     0.0312
年齢           13.0     0.1116
年齢           14.5     0.2604
年齢           20.0     0.4688
年齢           28.5     0.2812
年齢           33.5     0.1688
年齢           37.5     0.0938
年齢           47.5     0.0402


## 木を育てる

**深さ 1、葉 2 枚の木で正解率 1.0 に達します。** モデルをそのまま出力して読めるのが決定木の強みです。

In [5]:
tree = build_tree(points, labels)

print(tree)
print(f"深さ {depth(tree)}  葉の数 {leaf_count(tree)}  正解率 {accuracy(tree, points, labels):.2f}")

Node(split=Split(feature=1, threshold=20.0), left=Leaf(label=1), right=Leaf(label=0))
深さ 1  葉の数 2  正解率 1.00


## ジニとエントロピーは同じ木を作る

値の絶対値は違う（0.4688 と 0.9544）のに、**順序が同じなので選ばれる分割も同じ** になります。

In [6]:
by_gini = build_tree(points, labels, impurity=gini_impurity)
by_entropy = build_tree(points, labels, impurity=entropy)
print("同じ木か:", by_gini == by_entropy)

同じ木か: True


## 試してみる: 木の成長を止める

最大深さと最小サンプル数は、**第 4 章の正則化と同じ役割** です。木は放っておくと訓練データを丸暗記するまで育ちます。

In [7]:
for max_depth in [0, 1, 3]:
    t = build_tree(points, labels, max_depth=max_depth)
    print(f"max_depth={max_depth}  深さ {depth(t)}  葉 {leaf_count(t)}  "
          f"正解率 {accuracy(t, points, labels):.2f}")

max_depth=0  深さ 0  葉 1  正解率 0.62
max_depth=1  深さ 1  葉 2  正解率 1.00
max_depth=3  深さ 1  葉 2  正解率 1.00
